# Melbourne Water + Soil Data
## Initial Pre-processing (Shival)

#### Purpose

This notebook focuses on the initial data cleaning and preparation of the Melbourne Water Pipeline and Soil datasets for Project 3B.

The main goals are to:

- load and inspect the two raw datasets:
  - `Water_Supply_Main_Pipelines.csv`
  - `Soil-Readings-Melbourne.csv`
- understand the structure, data types, and data quality of each dataset
- clean and standardise key fields such as dates, numeric values, text fields, and IDs
- identify useful columns for later analysis and modelling our intial datasets
- check how the two datasets can be merged
- create a clean and consistent base dataset for the next stages of the project

In [8]:
import pandas as pd

In [9]:
#Initial check of current datasets to see what may need to be removed

#Load datasets into variables for cleaning and examination
water = pd.read_csv("../data/raw/Melbourne/Water_Supply_Main_Pipelines.csv")
soil = pd.read_csv("../data/raw/Melbourne/Soil-Readings-Melbourne.csv")

#print datasets to list and then proceed to print the first 5 rows of every column within said datasets.
print(water.columns.tolist())
print(soil.columns.tolist())

print(water.head())
print(soil.head())

['OBJECTID', 'MXUNITID', 'MXSITEID', 'COMPKEY', 'COMPTYPE', 'UNITID', 'UNITID2', 'PARALLEL_LINE_NBR', 'MAIN_LINE_TYPE', 'MAIN_CLASS', 'MAIN_NAME', 'ASSET_ID', 'SUBAREA', 'Shape', 'MATERIAL', 'UPSTREAM_IL', 'DOWNSTREAM_IL', 'PIPE_LENGTH', 'PIPE_WIDTH', 'SOIL_TYPE', 'DATE_RELINED', 'GIS_ID', 'FIELD_TEAM', 'WS_CONSTRUCTION_FILE', 'DATE_OF_CONSTRUCTION', 'EPMS_SEC_NO', 'ASCONST_PLAN_NO', 'SKETCH_PLAN_NO', 'SOURCE', 'METHOD_OF_CAPTURE', 'DATE_CAPTURED', 'DATE_LAST_UPDATED', 'SERVICE_STATUS', 'SERVICE_STATUS_CHG_DATE', 'SERVICE_STATUS_PLAN_NO', 'COMMENTS', 'MI_PRINX']
['Local_Time', 'Site_Name', 'date', 'time', 'Probe_Measure', 'Soil_Value', 'Unit']
   OBJECTID  MXUNITID  MXSITEID  COMPKEY  COMPTYPE         UNITID  \
0         1       NaN       NaN   296455         0   M474-00374BR   
1         2       NaN       NaN   293331         0       WZ060-ZA   
2         3       NaN       NaN  1017825         0  WZ060-01040BR   
3         4       NaN       NaN   234582  10273261       WR144V-R   
4  

## Cleaning
### Water Pipeline Dataset

In [10]:
#Remove unecessary unnamed columns

#Removes empty unnamed columns within water dataset
water = water.loc[:, ~water.columns.str.contains("^Unnamed")]

#Keeps relevant columns
water = water[
    [
        "ASSET_ID",              
        "MAIN_LINE_TYPE",
        "MAIN_CLASS",
        "MAIN_NAME",
        "MATERIAL",
        "PIPE_LENGTH",
        "PIPE_WIDTH",
        "SOIL_TYPE",
        "DATE_RELINED",
        "DATE_OF_CONSTRUCTION",
        "SERVICE_STATUS"
    ]
].copy()

#convert 'DATE_OF_CONSTRUCTION' to datetime for easier readability.
water["DATE_OF_CONSTRUCTION"] = pd.to_datetime(
    water["DATE_OF_CONSTRUCTION"], 
    format="%m/%d/%Y %I:%M:%S %p", #since our dataset contains a non standard format (MM/DD/YYYY + timestamp), we need to specify exact format for datetime to avoid runtime warnings
    errors="coerce" #this command serves to ensure that pandas doesn't crash when coming across invalid dates
)

#We can assume that the actual age of all of the varrying pipes within this data set are ages relative to their established date. Thus,
#we can simply subtract their build date from current year (2026) to estimate how old they are. We'll fit this data into a 'PIPE_AGE' column
water["PIPE_AGE"] = 2026 - water["DATE_OF_CONSTRUCTION"].dt.year

print(water.head())
print(water.isnull().sum())


  ASSET_ID MAIN_LINE_TYPE MAIN_CLASS                       MAIN_NAME MATERIAL  \
0     M474         MAINBG          T  ST GEORGES RD REPLACEMENT MAIN     MSCL   
1     Z060         MAINAG          S       CARDINIA CREEK MINI HYDRO     MSCL   
2     M281         BYPASS          S  CARDINIA-DAND'G-NOTTING HILL M     MSCL   
3     R144         MAINBG          S                 TYABB RESERVOIR       MS   
4     M281         BYPASS          S  CARDINIA-DAND'G-NOTTING HILL M     MSCL   

   PIPE_LENGTH  PIPE_WIDTH SOIL_TYPE DATE_RELINED DATE_OF_CONSTRUCTION  \
0         2.72        1400       NaN          NaN           2017-12-01   
1         1.88         225       NaN          NaN           2017-12-18   
2         1.74         225      SAND          NaN           2011-08-26   
3       131.30         100       NaN          NaN           2017-02-22   
4         5.26         225      SAND          NaN           2011-08-26   

  SERVICE_STATUS  PIPE_AGE  
0             IN       9.0  
1         

In [11]:
# save new processed dataset to new csv
water.to_csv("../data/processed/cleaned_water_data.csv", index=False)

### Soil Dataset

In [12]:
#The cleaning stage for the soil dataset is mostly the same as the water dataset, we'll begin by removing unammed columns
soil = soil.loc[:, ~soil.columns.str.contains("^Unnamed")]

print(soil.columns.tolist())
print(soil.isnull().sum())

['Local_Time', 'Site_Name', 'date', 'time', 'Probe_Measure', 'Soil_Value', 'Unit']
Local_Time       928
Site_Name          0
date             928
time             928
Probe_Measure      0
Soil_Value         0
Unit               0
dtype: int64


In [13]:
#using the above metric, we'll then verify the columns we'd like to keep and rename them to a more standardised naming convention
soil = soil.rename(columns={
    "Local_Time": "LOCAL_TIME",
    "Site_Name": "SITE_NAME",
    "date": "DATE",
    "time": "TIME",
    "Probe_Measure": "PROBE_MEASURE",
    "Soil_Value": "SOIL_VALUE",
    "Unit": "UNIT"
})

print(soil.head())
print(soil.isnull().sum())

                  LOCAL_TIME            SITE_NAME      DATE      TIME  \
0  2023-03-03T08:00:00+11:00  Princes Park Lawn 5  3/3/2023   8:00:00   
1  2023-03-03T08:00:00+11:00  Princes Park Lawn 5  3/3/2023   8:00:00   
2  2023-03-03T10:00:00+11:00  Princes Park Lawn 5  3/3/2023  10:00:00   
3  2023-03-03T06:00:00+11:00  Princes Park Lawn 5  3/3/2023   6:00:00   
4  2023-03-03T10:00:00+11:00  Princes Park Lawn 5  3/3/2023  10:00:00   

           PROBE_MEASURE  SOIL_VALUE  UNIT  
0  Soil Moisture 50cm #0       36.21  %VWC  
1  Soil Moisture 60cm #0       46.91  %VWC  
2  Soil Moisture 60cm #0       46.91  %VWC  
3  Soil Moisture 70cm #0       56.90  %VWC  
4  Soil Moisture 70cm #0       56.90  %VWC  
LOCAL_TIME       928
SITE_NAME          0
DATE             928
TIME             928
PROBE_MEASURE      0
SOIL_VALUE         0
UNIT               0
dtype: int64


## Dataset preparation for using the Soil Dataset as a feature to compliment the water dataset

In [14]:
# Since a direct merge isn't the best option given the difference of the datasets, we're instead going to go for an approach where
# we filter the soil data to use it as a supporting dataset for our Melbourne Water Main Dataset
soil_moisture = soil[
    soil["PROBE_MEASURE"].str.contains("Moisture", case=False, na=False)
    ]

#it's also important we convert the soil values itself to a numeric value for easier analysis and use in models
soil_moisture["SOIL_VALUE"] = pd.to_numeric(
    soil_moisture["SOIL_VALUE"], errors="coerce"
)

# for the best result, we'll use the mean function to calcuate the average soil moisture of the soil moisture 
# column in order to get a single value we can use for testing
avg_moisture = soil_moisture["SOIL_VALUE"].mean()
print("Average Soil Moisture:", avg_moisture)

# NOTE:
# While we calculated average soil moisture for exploratory analysis, assigning a single constant value to all water mains not provide 
# meaningful variation for predictive modelling. Therefore, we will not integrate this value directly into the dataset.
# Instead, we will use the existing SOIL_TYPE field as our primary environmental feature.

Average Soil Moisture: 30.591046075676182


We can now utilise the above value to compliment our water dataset during our Model

In [15]:
# Clean SOIL_TYPE for modelling
water["SOIL_TYPE"] = water["SOIL_TYPE"].str.upper().str.strip()
water["SOIL_TYPE"] = water["SOIL_TYPE"].fillna("UNKNOWN")

print(water["SOIL_TYPE"].value_counts())

SOIL_TYPE
UNKNOWN    4500
CLAY       3115
UNKN       2120
CRK        1227
SAND       1162
SNC         188
S&C          90
TOPS         80
SS&C         41
MS           41
MS&C         26
SLCL         23
C&GR         20
C&BS         15
SS           14
SLTS          6
AIR           5
CST           4
CONC          2
C&RC          1
Name: count, dtype: int64


In [16]:
print(water.head())

  ASSET_ID MAIN_LINE_TYPE MAIN_CLASS                       MAIN_NAME MATERIAL  \
0     M474         MAINBG          T  ST GEORGES RD REPLACEMENT MAIN     MSCL   
1     Z060         MAINAG          S       CARDINIA CREEK MINI HYDRO     MSCL   
2     M281         BYPASS          S  CARDINIA-DAND'G-NOTTING HILL M     MSCL   
3     R144         MAINBG          S                 TYABB RESERVOIR       MS   
4     M281         BYPASS          S  CARDINIA-DAND'G-NOTTING HILL M     MSCL   

   PIPE_LENGTH  PIPE_WIDTH SOIL_TYPE DATE_RELINED DATE_OF_CONSTRUCTION  \
0         2.72        1400   UNKNOWN          NaN           2017-12-01   
1         1.88         225   UNKNOWN          NaN           2017-12-18   
2         1.74         225      SAND          NaN           2011-08-26   
3       131.30         100   UNKNOWN          NaN           2017-02-22   
4         5.26         225      SAND          NaN           2011-08-26   

  SERVICE_STATUS  PIPE_AGE  
0             IN       9.0  
1         